# Batch analysis

Looks over finished batches. Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.
The same figures as files: `uv run python -m moran_graphs.plots <batch>`.
Asymmetric-lung sweeps (several `flow_ratio` values per size) have their own notebook, `notebooks/asymmetric_lungs.ipynb`.

Mean times are conditional on fixation. `fixation_prob` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
%load_ext autoreload
%autoreload 2  

import matplotlib.pyplot as plt

from moran_graphs.plots import (PROPERTIES, hover_scatter, load, print_batch, print_run_report, property_grid,
                                scatter, versus_random)

## Configuration

The only cell to edit. Several batches are stitched with a `batch` column; a graph in two batches stays two rows.

In [ ]:
BATCHES = [
    # "2026-10-14-respiratory-size-sweep-2",
"2026-11-02-precision-test-stage2",
    ]
R_VALUES = [1.1]  # the per-r figures need exactly one
DIRECTED = [False, True]  # [False] undirected only, [True] directed only

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

In [ ]:
results, hists = load(BATCHES)
print(f"{results['hash'].nunique()} graphs, r values {sorted(results['r'].unique().tolist())}")
# Random graphs stay whatever DIRECTED is: they are the undirected cloud every type is compared to.
keep = results["r"].isin(R_VALUES) & (results["is_directed"].isin(DIRECTED) | (results["category"] == "Random"))
results = results[keep]
varying = [p for p in PROPERTIES if results[p].dtype != bool and results[p].nunique() > 1]
results.groupby(["category", "is_directed", "r"])[["fixation_prob", "mean_fixation_time", "n_censored"]].agg(["count", "mean", "min", "max"])

## Monte Carlo precision

How well each (graph, r) cell is measured: runs, fixed runs, SE and the relative 95% CI half-width of both metrics, against N. The dashed line is the target half-width `stage_two_plan` sizes n for. Fixed runs set the precision of both metrics, so cells with a small fixation probability are the noisiest.

In [ ]:
from moran_graphs.plots import precision

fig = precision(results)

Fixation-time distributions per graph type and N (left) against the matched random cloud (right), one figure per r. Narrow, untailed distributions need few fixed runs for a precise mean; the numbers at the bottom are the median CV. Where several graphs share a type and N (shape sweeps), their runs are pooled, so the left half also includes differences between shapes.

In [ ]:
from moran_graphs.plots import time_violins

for r in sorted(results["r"].unique()):
    fig = time_violins(results, hists, r)

How much of each random cloud's spread is simulation noise (Monte Carlo variance over the observed variance between its graphs). Near 0 the middle-95% band measures structure; near 1 the band, and the stands-out test built on it, is mostly noise.

In [ ]:
from moran_graphs.plots import random_noise_share

fig = random_noise_share(results)

Stage-two batches only: each cell's estimate against its independent pilot estimate as a z-score. About 5% of cells should fall beyond 1.96; many more means the engine, the seeding or the SE is wrong. Draws nothing for batches without a pilot.

In [ ]:
from moran_graphs.plots import pilot_agreement

fig = pilot_agreement(results)

In [ ]:
results.columns

In [ ]:
results[["batch", "name", "category", "n_nodes", "n_edges", "fixation_prob", "fixation_prob_lo", "fixation_prob_hi",
      "fixation_prob_over_random", "mean_fixation_time", "mean_fixation_time_se",
      "mean_fixation_time_over_random", "n_random_matches", "n_censored"]].sort_values("fixation_prob", ascending=False).head()

## Fixation probability and time

Hover a point to see its graph and recipe.

In [ ]:
results.shape == results[results["mean_fixation_time"] > 0].shape

In [ ]:
n_runs = results["n_fixed"] + results["n_extinct"]
results["steps_mean_absorption"] = (results["n_fixed"] * results["mean_fixation_time"] + results["n_extinct"] * results["mean_extinction_time"]) / n_runs
results["steps_mean_absorption"]

In [ ]:
# Log x: mean times span 154 to 6e7 steps here, so on a linear axis almost every graph sits at 0.
hover_scatter(results, x="mean_fixation_time", y="fixation_prob", marker_by="r").update_xaxes(type="log")

In [ ]:
hover_scatter(results, "mean_fixation_time", "fixation_time_sd")

In [ ]:
if results["r"].nunique() > 1:
    hover_scatter(results, "r", "fixation_prob").show()

## Against the random cloud, across sizes

Each graph type as a curve over N, against the random graphs with the same N and E (grey: their mean and middle 95%, 100 per size). Ratio panels: the graph over the random mean (`*_over_random` from `load`), so above 1 the graph fixes more often or slower than random; the band is the graph's single value over the random middle 95%, so where it excludes 1 the graph stands out.
A type needs a cloud at every size and one graph per size, the rest is printed. The complete graph has no cloud (no other graph has its N and E); its exact theory is the dashed well-mixed line.
The random graphs are undirected, so for directed types the cloud matches N and E but not the edge directions.

Thick pale red line: the theory formula from `theory.py` (written out in the legend), the anchor the simulated points should sit on within their error bars. Fixation probability and time for the cycle (both directions), directed line and directed star; fixation probability only for the star (no time formula yet) and for the directed single-root trees (line, star, mammalian, fish: only a mutant on the root fixes, so fixation probability = 1/N whatever r).

In [ ]:
animals = ["Mammalian lung", "Asymmetric lung", "Avian lung", "Fish gill"]  # the category labels; the rest are classical graphs

is_animal = results["category"].isin(animals)
is_random = results["category"] == "Random"
for directed in DIRECTED:
    for name, graphs in (("Animal", is_animal), ("Non-animal", ~is_animal & ~is_random)):
        fig = versus_random(results[graphs | is_random], directed)
        fig.suptitle(f"{name} graphs. {fig.get_suptitle()}")

## Structural properties

One panel per property. Look for properties where the respiratory graphs sit outside the random cloud.

In [ ]:
property_grid(results, "fixation_prob", varying).suptitle(f"r = {R_VALUES}");

In [ ]:
property_grid(results, "mean_fixation_time", varying).suptitle(f"r = {R_VALUES}");

## Ad hoc

Any two columns of `results`, or a subset of it. `hover=[...]` picks the columns shown on hover (default N, E, r, batch, recipe). `scatter(ax, df, x, y)` draws the same thing on a matplotlib axis.

In [ ]:
hover_scatter(results, "degree_std", "average_shortest_path_length")